![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Turning one flat file into a relational database: clean the data, split it into tables, wire up the keys, load it into SQLite.

> **Done when:** `data/project.db` exists with 3+ related tables, `sql/schema.sql` matches your ERD, no foreign key is dangling, and the row counts are what you expect.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

In [2]:
listings_raw = pd.read_csv("../data/raw/listings.csv.gz")
reviews_raw = pd.read_csv("../data/raw/reviews.csv.gz",
                          usecols=["id", "listing_id", "date", "reviewer_id"])

print(f"listings_raw: {listings_raw.shape[0]:,} rows × {listings_raw.shape[1]} columns")
print(f"reviews_raw:  {reviews_raw.shape[0]:,} rows × {reviews_raw.shape[1]} columns")

listings_raw: 15,293 rows × 90 columns
reviews_raw:  1,033,523 rows × 4 columns


---
## 2. Clean

Work through the problems you listed in notebook 01. Every decision you make here — drop the nulls or keep them, cast or discard — is one you may be asked to defend on Friday, so note why.

In [3]:
cols_to_keep = [
    "id", "host_id", "host_name", "host_is_superhost", "host_identity_verified",
    "host_listings_count", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "room_type", "accommodates", "price", "minimum_nights", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "estimated_occupancy_l365d",
    "review_scores_rating", "license", "source",
]

listings = listings_raw[cols_to_keep].copy()
print(f"columns: {listings_raw.shape[1]} → {listings.shape[1]}")

columns: 90 → 19


In [4]:
listings = listings.rename(columns={
    "id": "listing_id",
    "neighbourhood_cleansed": "neighbourhood_name",
    "neighbourhood_group_cleansed": "district_name",
    "license": "licence_raw",
})

In [5]:
listings["price_eur"] = listings["price"].str.replace(r"[$,]", "", regex=True).astype(float)
listings = listings.drop(columns="price")

listings["price_eur"].describe()

count    13355.000000
mean       240.616470
std        339.474553
min          3.500000
25%         89.290000
50%        177.670000
75%        285.500000
max      10542.220000
Name: price_eur, dtype: float64

In [6]:
for col in ["host_is_superhost", "host_identity_verified"]:
    listings[col] = listings[col].map({"t": 1, "f": 0}).astype("Int64")

listings["is_current_scrape"] = (listings["source"] == "city scrape").astype(int)
listings = listings.drop(columns="source")

listings[["host_is_superhost", "host_identity_verified", "is_current_scrape"]].value_counts(dropna=False).head()

host_is_superhost  host_identity_verified  is_current_scrape
0                  1                       1                    8947
1                  1                       1                    3624
0                  1                       0                    1878
                   0                       1                     439
1                  1                       0                     282
Name: count, dtype: int64

In [7]:
lic = listings["licence_raw"]

conditions = [
    lic.isna(),
    lic.str.contains("Exempt - seasonal", na=False),
    lic.str.contains(r"HUTB-?\d", na=False),
    lic.str.contains("Exempt", na=False),
]
labels = ["missing", "exempt_seasonal", "hutb_licence", "exempt_other"]
listings["licence_status"] = np.select(conditions, labels, default="other_code")

# Normalise the HUTB number to one format: "HUTB-" + 6 digits
hutb_digits = lic.str.extract(r"HUTB-?(\d{6})")[0]
listings["licence_number"] = np.where(listings["licence_status"] == "hutb_licence",
                                      "HUTB-" + hutb_digits, None)
listings = listings.drop(columns="licence_raw")

print(listings["licence_status"].value_counts(), "\n")
print(listings["licence_number"].value_counts().head())

licence_status
hutb_licence       7406
missing            3043
exempt_seasonal    2960
other_code         1159
exempt_other        725
Name: count, dtype: int64 

licence_number
HUTB-000000    35
HUTB-012407    30
HUTB-012457    29
HUTB-246789    26
HUTB-007110    24
Name: count, dtype: int64


In [8]:
reviews = reviews_raw.rename(columns={"id": "review_id", "date": "review_date"})
reviews["review_date"] = pd.to_datetime(reviews["review_date"], format="%Y-%m-%d")

reviews.dtypes

listing_id              int64
review_id               int64
review_date    datetime64[us]
reviewer_id             int64
dtype: object

In [9]:
listings.isna().sum()[lambda s: s > 0]

host_name                   87
host_is_superhost           87
host_identity_verified      87
host_listings_count         87
minimum_nights               1
review_scores_rating      3548
price_eur                 1938
licence_number            7887
dtype: int64

In [10]:
listings[listings["minimum_nights"].isna()].T

,3520
listing_id,29047395
host_id,158023606
host_name,Habitat Apartments
host_is_superhost,0
host_identity_verified,1
host_listings_count,134.0
neighbourhood_name,la Dreta de l'Eixample
district_name,Eixample
room_type,Entire home/apt
accommodates,5


### Cleaning decisions

| Step | Decision | Why |
|---|---|---|
| Columns | 90 → 19 kept | Only what Q1, Q2 and the schema need; the 12 empty columns go with the rest |
| Names | `id` → `listing_id`, district/neighbourhood renamed | Avoid ambiguous columns in joins; use domain names |
| `price` | Text → float `price_eur` | Stored as "$409.00" but quoted in EUR |
| Booleans | "t"/"f" → 1/0 (`Int64`) | Countable and averageable in SQL; nullable integer keeps the 87 nulls |
| `source` | → `is_current_scrape` (1/0) | A flag named after its meaning |
| `license` | → `licence_status` (5 categories) + `licence_number` | Normalised to "HUTB-" + 6 digits so the same licence isn't counted twice |
| `review_date` | Text → datetime | Needed for "last 12 months" |
| Nulls kept | `price_eur` (1,938), `review_scores_rating` (3,548), `licence_number` (7,887), host attributes (87), `minimum_nights` (1) | Each null has a meaning; filling would invent data |
| Flagged, not changed | `HUTB-000000` on 35 listings | Placeholder licence: analysed in Q1, not cleaned away |

---
## 3. Design your tables

Decide what your tables are and how they relate before you write any code. Draw the ERD (Excalidraw, draw.io), export it as an image, commit it, and link it from your README.

Three tables minimum, with real primary and foreign keys.

### ERD

![ERD](../images/erd.png)

### Tables

| Table | Grain | Rows | Primary key | References |
|---|---|---|---|---|
| `districts` | one district | 10 | `district_id` | — |
| `neighbourhoods` | one neighbourhood | 69 | `neighbourhood_id` | `districts` |
| `hosts` | one host | 4,595 | `host_id` | — |
| `licence_status` | one licence status | 5 | `licence_status_id` | — |
| `listings` | one listing | 15,293 | `listing_id` | `hosts`, `neighbourhoods`, `licence_status` |
| `reviews` | one review | ~1.03M (after removing orphans) | `review_id` | `listings` |

### Design decisions

- **Two-level location hierarchy** (`districts → neighbourhoods → listings`): each neighbourhood belongs to exactly one district (checked in notebook 01, 5.3). Q1 compares districts, so the district is a real entity, not a label repeated 15,293 times.
- **`hosts` as its own table:** every host attribute has a single value per host (notebook 01, 5.4), so storing it on each listing would repeat it and risk inconsistencies.
- **`licence_status` as a lookup table:** 5 categories derived from a messy text field; the core variable of Q1. Its `description` explains each code, so queries and charts don't depend on remembering what `other_code` means.
- **`licence_number` stays in `listings`:** each listing displays its own number, so it belongs to the listing, not to the status.
- **Keys:** `listing_id`, `host_id` and `review_id` are Airbnb's own ids (natural keys, already unique). `district_id`, `neighbourhood_id` and `licence_status_id` are integer ids created here (surrogate keys), because the source data only has names.

### Types

- **Integers** for ids, counts and 1/0 flags.
- **`price_eur` as `REAL`:** SQLite has no true `DECIMAL` type. It is an asking price used for medians and comparisons, not accounting, so floating-point precision is enough.
- **`review_date` as `TEXT` in ISO format (`YYYY-MM-DD`):** SQLite has no `DATE` type. ISO text sorts correctly and works with SQLite's date functions.
- **Nullable on purpose:** `price_eur`, `minimum_nights`, `review_scores_rating`, `licence_number` and the host attributes, for the reasons documented in section 2.

### Rejected

- **`room_type` table (4 values)** and **`property_type` table (50 values, mostly repeating `room_type`):** a join that adds no information.
- **A `reviewers` table:** only `reviewer_id` is kept (no names, for privacy), and 95% of reviewers left a single review, so it would be a one-column table.

---
## 4. Build the tables

Split the cleaned data into the tables you designed.

In [11]:
districts = (listings[["district_name"]]
             .drop_duplicates()
             .sort_values("district_name")
             .reset_index(drop=True))
districts.insert(0, "district_id", districts.index + 1)

districts

,district_id,district_name
0,1,Ciutat Vella
1,2,Eixample
2,3,Gràcia
3,4,Horta-Guinardó
4,5,Les Corts
5,6,Nou Barris
6,7,Sant Andreu
7,8,Sant Martí
8,9,Sants-Montjuïc
9,10,Sarrià-Sant Gervasi


In [12]:
neighbourhoods = (listings[["neighbourhood_name", "district_name"]]
                  .drop_duplicates()
                  .sort_values("neighbourhood_name")
                  .reset_index(drop=True))
neighbourhoods.insert(0, "neighbourhood_id", neighbourhoods.index + 1)

# Replace the district name with its id (the foreign key)
neighbourhoods = (neighbourhoods
                  .merge(districts, on="district_name", how="left", validate="many_to_one")
                  .drop(columns="district_name"))

print(neighbourhoods.shape)
neighbourhoods.head()

(69, 3)


,neighbourhood_id,neighbourhood_name,district_id
0,1,Can Baró,4
1,2,Ciutat Meridiana,6
2,3,Diagonal Mar i el Front Marítim del Poblenou,8
3,4,Horta,4
4,5,Hostafrancs,9


In [13]:
hosts = (listings[["host_id", "host_name", "host_is_superhost",
                   "host_identity_verified", "host_listings_count"]]
         .drop_duplicates()
         .rename(columns={"host_is_superhost": "is_superhost",
                          "host_identity_verified": "identity_verified"}))

print(hosts.shape)
print("duplicated host_id:", hosts["host_id"].duplicated().sum())
hosts.isna().sum()

(4595, 5)
duplicated host_id: 0


host_id                 0
host_name              42
is_superhost           42
identity_verified      42
host_listings_count    42
dtype: int64

In [14]:
licence_status = pd.DataFrame({
    "licence_status_id": [1, 2, 3, 4, 5],
    "status_code": ["hutb_licence", "exempt_seasonal", "exempt_other", "other_code", "missing"],
    "description": [
        "Barcelona tourist-home licence number shown (HUTB)",
        "Declared exempt as a seasonal rental (31+ nights)",
        "Declared exempt for another reason (e.g. hostel, hotel)",
        "Other registration code (hotel, hostel or unrecognised format)",
        "No licence information shown",
    ],
})

licence_status

,licence_status_id,status_code,description
0,1,hutb_licence,Barcelona tourist-home licence number shown (H...
1,2,exempt_seasonal,Declared exempt as a seasonal rental (31+ nights)
2,3,exempt_other,Declared exempt for another reason (e.g. hoste...
3,4,other_code,"Other registration code (hotel, hostel or unre..."
4,5,missing,No licence information shown


In [15]:
listings_table = (listings
    .merge(neighbourhoods[["neighbourhood_id", "neighbourhood_name"]],
           on="neighbourhood_name", how="left", validate="many_to_one")
    .merge(licence_status[["licence_status_id", "status_code"]],
           left_on="licence_status", right_on="status_code", how="left", validate="many_to_one")
)

listings_table = listings_table[[
    "listing_id", "host_id", "neighbourhood_id", "licence_status_id", "licence_number",
    "room_type", "accommodates", "price_eur", "minimum_nights", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "estimated_occupancy_l365d",
    "review_scores_rating", "is_current_scrape",
]]

print(listings_table.shape)
listings_table.head(3)

(15293, 15)


,listing_id,host_id,neighbourhood_id,licence_status_id,licence_number,room_type,accommodates,price_eur,minimum_nights,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,review_scores_rating,is_current_scrape
0,18674,71615,57,1,HUTB-002062,Entire home/apt,8,409.00,1.0,123,59,9,54,4.41,1
1,23197,90417,28,1,HUTB-005057,Entire home/apt,5,388.00,2.0,239,99,11,66,4.84,1
2,34981,73163,27,1,HUTB-001506,Entire home/apt,9,397.13,1.0,329,304,37,222,4.59,1


In [16]:
reviews_table = reviews[["review_id", "listing_id", "review_date", "reviewer_id"]]
print(reviews_table.shape)

(1033523, 4)


### Tables built

| Table | Rows | Notes |
|---|---|---|
| `districts` | 10 | Ids assigned alphabetically |
| `neighbourhoods` | 69 | Each points to one district |
| `hosts` | 4,595 | 42 hosts with no profile data (87 listings); kept with null attributes |
| `licence_status` | 5 | Built by hand from the parsing rules, with a description per code |
| `listings` | 15,293 | Row count unchanged after the merges |
| `reviews` | 1,033,523 | Orphans still included: removed and verified in section 5 |

All merges use `validate="many_to_one"`, so a duplicated key in a parent table would raise an error instead of silently multiplying rows.

---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

In [17]:
primary_keys = {
    "districts": (districts, "district_id"),
    "neighbourhoods": (neighbourhoods, "neighbourhood_id"),
    "hosts": (hosts, "host_id"),
    "licence_status": (licence_status, "licence_status_id"),
    "listings": (listings_table, "listing_id"),
    "reviews": (reviews_table, "review_id"),
}

for name, (table, pk) in primary_keys.items():
    print(f"{name:15} duplicated: {table[pk].duplicated().sum():>3}   null: {table[pk].isna().sum():>3}")

districts       duplicated:   0   null:   0
neighbourhoods  duplicated:   0   null:   0
hosts           duplicated:   0   null:   0
licence_status  duplicated:   0   null:   0
listings        duplicated:   0   null:   0
reviews         duplicated:   0   null:   0


In [18]:
foreign_keys = [
    # (child table name, child df, FK column, parent df, PK column)
    ("neighbourhoods", neighbourhoods, "district_id", districts, "district_id"),
    ("listings", listings_table, "host_id", hosts, "host_id"),
    ("listings", listings_table, "neighbourhood_id", neighbourhoods, "neighbourhood_id"),
    ("listings", listings_table, "licence_status_id", licence_status, "licence_status_id"),
    ("reviews", reviews_table, "listing_id", listings_table, "listing_id"),
]

for child_name, child, fk, parent, pk in foreign_keys:
    orphans = (~child[fk].isin(parent[pk])).sum()
    nulls = child[fk].isna().sum()
    print(f"{child_name}.{fk:18} orphans: {orphans:>5,}   nulls: {nulls}")

neighbourhoods.district_id        orphans:     0   nulls: 0
listings.host_id            orphans:     0   nulls: 0
listings.neighbourhood_id   orphans:     0   nulls: 0
listings.licence_status_id  orphans:     0   nulls: 0
reviews.listing_id         orphans: 6,592   nulls: 0


In [19]:
orphan_reviews = reviews_table[~reviews_table["listing_id"].isin(listings_table["listing_id"])]

print("orphan reviews:         ", len(orphan_reviews))
print("distinct missing listings:", orphan_reviews["listing_id"].nunique())
print("most recent orphan review:", orphan_reviews["review_date"].max().date())

orphan reviews:          6592
distinct missing listings: 77
most recent orphan review: 2026-07-01


In [20]:
rows_before = len(reviews_table)
reviews_table = reviews_table[reviews_table["listing_id"].isin(listings_table["listing_id"])]

print(f"reviews: {rows_before:,} → {len(reviews_table):,} (removed {rows_before - len(reviews_table):,})")
print("remaining orphans:", (~reviews_table["listing_id"].isin(listings_table["listing_id"])).sum())

reviews: 1,033,523 → 1,026,931 (removed 6,592)
remaining orphans: 0


### Referential integrity

- **Primary keys:** unique and non-null in all six tables.
- **Foreign keys:** no orphans and no nulls in `neighbourhoods.district_id`, `listings.host_id`, `listings.neighbourhood_id` and `listings.licence_status_id`.
- **Orphan reviews:** 6,592 reviews (0.6%) pointed to 77 listings that are not in the listings snapshot. The most recent was left on 1 July 2026, so they are not just old deleted listings: the reviews file was scraped a few days after the listings file, and some listings were removed or added in between.
- **Decision: removed.** A review with no listing cannot be attributed to a district, host or licence status, so it answers neither research question, and keeping it would break the foreign key. Reviews go from 1,033,523 to **1,026,931**.
- **Effect on the analysis:** 77 listings out of 15,293 (0.5%) is too small to change any district-level result.

---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

In [21]:
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/project.db")
SCHEMA_PATH = Path("../sql/schema.sql")

DB_PATH.unlink(missing_ok=True)         

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON")
con.executescript(SCHEMA_PATH.read_text())

print("tables created:", [row[0] for row in con.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table'")])

tables created: ['districts', 'licence_status', 'hosts', 'neighbourhoods', 'listings', 'reviews']


In [22]:
reviews_table = reviews_table.copy()
reviews_table["review_date"] = reviews_table["review_date"].dt.strftime("%Y-%m-%d")

tables_in_load_order = {
    "districts": districts,
    "neighbourhoods": neighbourhoods,
    "hosts": hosts,
    "licence_status": licence_status,
    "listings": listings_table,
    "reviews": reviews_table,
}

for name, table in tables_in_load_order.items():
    table.to_sql(name, con, if_exists="append", index=False)
    print(f"loaded {name:15} {len(table):>9,} rows")

con.commit()

loaded districts              10 rows
loaded neighbourhoods         69 rows
loaded hosts               4,595 rows
loaded licence_status          5 rows
loaded listings           15,293 rows
loaded reviews         1,026,931 rows


In [23]:
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name, table in tables_in_load_order.items():
    path = CLEAN_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"saved {path.name:20} {len(table):>9,} rows")

saved districts.csv               10 rows
saved neighbourhoods.csv          69 rows
saved hosts.csv                4,595 rows
saved licence_status.csv           5 rows
saved listings.csv            15,293 rows
saved reviews.csv          1,026,931 rows


### Database created

- `data/project.db` is rebuilt from nothing on every run: the old file is deleted, then `sql/schema.sql` creates the tables, constraints and indexes.
- Foreign keys are **enforced** (`PRAGMA foreign_keys = ON`), so a child row pointing to a missing parent is rejected instead of loaded.
- Tables are loaded **parents first** (districts → neighbourhoods → hosts → licence_status → listings → reviews).
- Data is appended into the tables created by `schema.sql` (`if_exists="append"`), so the column types and constraints are the ones designed in section 3, not the ones pandas would infer.

---
## 7. Verify

Row counts per table, and one join that returns the rows you expect. Then open `data/project.db` in DB Browser and look at it.

In [24]:
for name, table in tables_in_load_order.items():
    db_rows = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    status = "OK" if db_rows == len(table) else "MISMATCH"
    print(f"{name:15} dataframe: {len(table):>9,}   database: {db_rows:>9,}   {status}")

districts       dataframe:        10   database:        10   OK
neighbourhoods  dataframe:        69   database:        69   OK
hosts           dataframe:     4,595   database:     4,595   OK
licence_status  dataframe:         5   database:         5   OK
listings        dataframe:    15,293   database:    15,293   OK
reviews         dataframe: 1,026,931   database: 1,026,931   OK


In [25]:
print("foreign keys enforced:", con.execute("PRAGMA foreign_keys").fetchone()[0] == 1)
print("broken foreign keys:  ", con.execute("PRAGMA foreign_key_check").fetchall())
print("integrity check:      ", con.execute("PRAGMA integrity_check").fetchone()[0])

foreign keys enforced: True
broken foreign keys:   []
integrity check:       ok


In [26]:
full_join = """
SELECT COUNT(*) AS n_rows
FROM listings AS l
JOIN neighbourhoods AS n  ON l.neighbourhood_id  = n.neighbourhood_id
JOIN districts AS d       ON n.district_id       = d.district_id
JOIN hosts AS h           ON l.host_id           = h.host_id
JOIN licence_status AS ls ON l.licence_status_id = ls.licence_status_id
"""
pd.read_sql(full_join, con)

,n_rows
0,15293


In [27]:
pd.read_sql("SELECT COUNT(*) AS n_rows FROM reviews AS r JOIN listings AS l ON r.listing_id = l.listing_id", con)

,n_rows
0,1026931


In [28]:
licence_by_district = """
SELECT d.district_name,
       COUNT(*) AS n_listings,
       SUM(CASE WHEN ls.status_code = 'missing' THEN 1 ELSE 0 END) AS n_no_licence
FROM listings AS l
JOIN neighbourhoods AS n  ON l.neighbourhood_id  = n.neighbourhood_id
JOIN districts AS d       ON n.district_id       = d.district_id
JOIN licence_status AS ls ON l.licence_status_id = ls.licence_status_id
GROUP BY d.district_name
ORDER BY n_listings DESC
"""
sql_result = pd.read_sql(licence_by_district, con)
sql_result

,district_name,n_listings,n_no_licence
0,Eixample,5835,864
1,Ciutat Vella,3247,794
2,Sants-Montjuïc,1448,285
3,Sant Martí,1436,359
4,Gràcia,1366,310
5,Sarrià-Sant Gervasi,888,149
6,Horta-Guinardó,358,106
7,Les Corts,327,56
8,Sant Andreu,225,66
9,Nou Barris,163,54


In [29]:
# Same numbers computed from the cleaned DataFrame, without the database
pandas_result = (listings.groupby("district_name")
                 .agg(n_listings=("listing_id", "size"),
                      n_no_licence=("licence_status", lambda s: (s == "missing").sum()))
                 .sort_values("n_listings", ascending=False)
                 .reset_index())

print("SQL and pandas agree:", sql_result.equals(pandas_result))

SQL and pandas agree: True


In [30]:
con.close()

### Verification

- **Row counts** in the database match the DataFrames for all six tables.
- **Foreign keys** are enforced (`PRAGMA foreign_keys = 1`), and SQLite's own `PRAGMA foreign_key_check` finds no broken key. This confirms the pandas check from section 5 with an independent one.
- **Joins lose no rows:** listings joined with all four parent tables returns all 15,293 listings, and reviews joined with listings returns all 1,026,931 reviews.
- **Cross-check:** the number of listings and of listings without a licence per district is identical whether computed in SQL on the database or in pandas on the cleaned data.
- Opened in DB Browser: six tables, five indexes, three foreign keys on `listings`.